# Use the RDF Data of the Swiss Political Parties

In [1]:
# load graph from the Turtle file
from rdflib import Graph

g = Graph()
g.parse("../data/output/parties.ttl", format="turtle")

<Graph identifier=N7564568a0f8849db9842da3904d029a0 (<class 'rdflib.graph.Graph'>)>

In [2]:
import pandas as pd

def sparql_to_df(graph: Graph, query: str) -> pd.DataFrame:
    result = graph.query(query)
    return pd.DataFrame(
        [
            [val.toPython() if val is not None else None for val in row]
            for row in result
        ],
        columns=[str(var) for var in result.vars]
    )

## Overview of Identities

In [3]:
query = """

PREFIX schema: <http://schema.org/>
PREFIX vl: <https://version.link/>
PREFIX pol: <https://politics.schema.ld.admin.ch/party-vocab/>

SELECT ?id_int ?founding_date ?dissolution_date ?name_de ?abbr_de ?name_fr ?abbr_fr ?name_it ?abbr_it ?parent_id WHERE {
    
    ?party a vl:Identity ;
        schema:identifier ?id ;
        schema:validFrom ?founding_date .

    OPTIONAL {
        ?party schema:validThrough ?dissolution_date .
    }

    # german name and abbreviation
    OPTIONAL {
        ?party schema:name ?name_de .
        FILTER (lang(?name_de) = "de")
    }
    OPTIONAL {
        ?party schema:alternateName ?abbr_de .
        FILTER (lang(?abbr_de) = "de")
    }

    # french name and abbreviation
    OPTIONAL {
        ?party schema:name ?name_fr .
        FILTER (lang(?name_fr) = "fr")
    }
    OPTIONAL {
        ?party schema:alternateName ?abbr_fr .
        FILTER (lang(?abbr_fr) = "fr")
    }

    # italian name and abbreviation
    OPTIONAL {
        ?party schema:name ?name_it .
        FILTER (lang(?name_it) = "it")
    }
    OPTIONAL {
        ?party schema:alternateName ?abbr_it .
        FILTER (lang(?abbr_it) = "it")
    }

    OPTIONAL {
        ?party pol:childOf ?parent .
        ?parent schema:identifier ?parent_id .
    }

    BIND(xsd:integer(?id) AS ?id_int)

} ORDER BY ?id_int

"""

df = sparql_to_df(g, query)
df.to_csv("../data/output/parties_overview.csv", index=False, encoding="utf-8")
display(df)

,id_int,founding_date,dissolution_date,name_de,abbr_de,name_fr,abbr_fr,name_it,abbr_it,parent_id
0,1,1971-09-22,None,Schweizerische Volkspartei,SVP,Union Démocratique du Centre,UDC,NaN,NaN,NaN
1,2,1888-10-21,None,Sozialdemokratische Partei der Schweiz,SP Schweiz,Parti socialiste suisse,PS Suisse,NaN,NaN,NaN
2,3,2009-01-01,None,FDP.Die Liberalen Schweiz,NaN,PLR.Les Libéraux-Radicaux,NaN,NaN,NaN,NaN
3,4,2021-01-01,None,Die Mitte Schweiz,Mitte,Le Centre,NaN,NaN,NaN,NaN
4,5,1983-05-28,None,GRÜNE Schweiz,GRÜNE,Les VERT-E-S suisses,VERT-E-S,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
417,421,1943-09-15,1990-06-30,Freiwirtschaftliche Bewegung der Schweiz,NaN,NaN,NaN,NaN,NaN,NaN
418,422,1994-06-04,2011-12-31,Demokratisch-Soziale Partei der Schweiz,DSP der Schweiz,NaN,NaN,NaN,NaN,NaN
419,423,1995-09-23,2009-12-31,Demokratisch-Soziale Partei Baselland,NaN,NaN,NaN,NaN,NaN,422
420,424,1987-11-03,2016-12-31,Demokratisch-Soziale Partei Graubünden,DSP Graubünden,NaN,NaN,NaN,NaN,422


## Get Info via Specific Date and Identity

In [4]:
query = """

PREFIX vl: <https://version.link/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX schema: <http://schema.org/>

SELECT ?identity ?date ?version ?name WHERE {
    BIND ("1990-01-01"^^xsd:date AS ?date)
    BIND (<https://politics.ld.admin.ch/party-id/5> AS ?identity)
    ?version vl:identity ?identity ;
        schema:name ?name ;
        schema:validFrom ?validFrom .

    OPTIONAL {
        ?version schema:validThrough ?validThrough .
    }

    FILTER (?validFrom <= ?date && (!bound(?validThrough) || ?validThrough >= ?date))
}

"""

df = sparql_to_df(g, query)
display(df)

,identity,date,version,name
0,https://politics.ld.admin.ch/party-id/5,1990-01-01,https://politics.ld.admin.ch/party-version/5_1...,Grüne Partei der Schweiz
1,https://politics.ld.admin.ch/party-id/5,1990-01-01,https://politics.ld.admin.ch/party-version/5_1...,Parti écologiste suisse


## Number of Versions per Party

In [5]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?party (COUNT(?version) AS ?count) WHERE {
    ?party a vl:Identity .
    ?version vl:identity ?party .
} GROUP BY ?party ORDER BY DESC(?count)

"""

df = sparql_to_df(g, query)
display(df)

,party,count
0,https://politics.ld.admin.ch/party-id/128,11
1,https://politics.ld.admin.ch/party-id/311,9
2,https://politics.ld.admin.ch/party-id/35,8
3,https://politics.ld.admin.ch/party-id/37,8
4,https://politics.ld.admin.ch/party-id/103,7
...,...,...
415,https://politics.ld.admin.ch/party-id/13,1
416,https://politics.ld.admin.ch/party-id/8,1
417,https://politics.ld.admin.ch/party-id/6,1
418,https://politics.ld.admin.ch/party-id/3,1


## All Names of a Party

Caution: This query will return the names of all versions. If the name is the same for multiple versions, it will be returned multiple times.

In [6]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?name ?validFrom ?validThrough WHERE {
    ?version vl:identity <https://politics.ld.admin.ch/party-id/5> ;
        schema:name ?name ;
        schema:validFrom ?validFrom .

    OPTIONAL {
        ?version schema:validThrough ?validThrough .
    }
} ORDER BY ?validFrom
    
"""

df = sparql_to_df(g, query)
display(df)

,name,validFrom,validThrough
0,Föderation der grünen Parteien der Schweiz,1983-05-28,1986-06-30
1,Fédération des partis écologistes de Suisse,1983-05-28,1986-06-30
2,Grüne Partei der Schweiz,1986-07-01,1993-09-04
3,Parti écologiste suisse,1986-07-01,1993-09-04
4,Grüne – Grüne Partei der Schweiz,1993-09-05,2007-07-18
5,Les Verts - Parti écologiste suisse,1993-09-05,2007-07-18
6,Grüne – Grüne Partei der Schweiz,2007-07-19,2018-06-30
7,Les Verts - Parti écologiste suisse,2007-07-19,2018-06-30
8,Grüne – Grüne Partei der Schweiz,2018-07-01,2021-03-26
9,Les Verts - Parti écologiste suisse,2018-07-01,2021-03-26


## All Parties with their Founding Date

In [7]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?name ?founding WHERE {

    ?party a vl:Identity ;
        schema:validFrom ?founding ;
        schema:name ?name .

    FILTER(lang(?name) = "de")

} ORDER BY ?founding

"""

df = sparql_to_df(g, query)
display(df)

,name,founding
0,FDP.Die Liberalen Luzern,1831-07-01
1,FDP.Die Liberalen Kanton Schwyz,1833-07-01
2,Oberegger Verein,1834-10-24
3,Die Mitte Kanton St.Gallen,1834-12-28
4,Verband Thurgauer Landwirtschaft,1835-07-01
...,...,...
302,Evangelische Volkspartei Kanton Zug,2021-12-09
303,Die Mitte Basel-Landschaft,2022-01-01
304,Grünliberale Partei Kanton Uri,2022-04-01
305,Partei der Arbeit Aargau,2022-07-01


## All Relationships of a Party Version

In [8]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?version ?from ?name WHERE {
    <https://politics.ld.admin.ch/party-version/1_1971-09-22> (vl:successor|vl:predecessor)+ ?version .

    ?version schema:validFrom ?from .

    OPTIONAL {
        ?version schema:name ?name .
        FILTER (lang(?name) = "de")
    }

} ORDER BY ?from
    
"""

df = sparql_to_df(g, query)
display(df)

,version,from,name
0,https://politics.ld.admin.ch/party-version/22_...,1840-11-05,Vorläufer der CVP
1,https://politics.ld.admin.ch/party-version/16_...,1880-01-01,Vorläufer der Demokraten
2,https://politics.ld.admin.ch/party-version/17_...,1905-03-19,Schweizerische Demokratische Partei
3,https://politics.ld.admin.ch/party-version/10_...,1912-04-22,Schweizerische Konservative Volkspartei
4,https://politics.ld.admin.ch/party-version/209...,1920-01-01,"Vorläufer der Bauern-, Gewerbe und Bürgerpartei"
5,https://politics.ld.admin.ch/party-version/11_...,1936-12-23,"Bauern-, Gewerbe- und Bürgerpartei"
6,https://politics.ld.admin.ch/party-version/18_...,1941-07-01,Demokratische Partei der Schweiz
7,https://politics.ld.admin.ch/party-version/10_...,1957-07-01,Konservativ-Christlichsoziale Volkspartei
8,https://politics.ld.admin.ch/party-version/10_...,1970-07-01,NaN
9,https://politics.ld.admin.ch/party-version/10_...,1970-12-12,Christlichdemokratische Volkspartei der Schweiz


## Visualization of the Relationships of a Party Version

In [9]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target ?source_name ?target_name WHERE {
    
    <https://politics.ld.admin.ch/party-version/1_2008-11-01> (vl:successor|vl:predecessor)+ ?source .

    ?source vl:successor ?target .
        
    OPTIONAL {
        ?source schema:name ?source_name .
        FILTER (lang(?source_name) = "de")
    }
    OPTIONAL {
        ?target schema:name ?target_name .
        FILTER (lang(?target_name) = "de")
    }
}
    
"""

df = sparql_to_df(g, query)
display(df)

,source,target,source_name,target_name
0,https://politics.ld.admin.ch/party-version/1_1...,https://politics.ld.admin.ch/party-version/13_...,Schweizerische Volkspartei,Bürgerlich-Demokratische Partei Schweiz
1,https://politics.ld.admin.ch/party-version/1_1...,https://politics.ld.admin.ch/party-version/1_2...,Schweizerische Volkspartei,Schweizerische Volkspartei
2,https://politics.ld.admin.ch/party-version/13_...,https://politics.ld.admin.ch/party-version/4_2...,Bürgerlich-Demokratische Partei Schweiz,Die Mitte Schweiz
3,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/4_2...,Christlichdemokratische Volkspartei der Schweiz,Die Mitte Schweiz
4,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...,Christlichdemokratische Volkspartei der Schweiz,Christlichdemokratische Volkspartei der Schweiz
5,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/66_...,Christlichdemokratische Volkspartei der Schweiz,Christlich-Soziale Partei der Schweiz
6,https://politics.ld.admin.ch/party-version/66_...,https://politics.ld.admin.ch/party-version/66_...,Christlich-Soziale Partei der Schweiz,Mitte Links – Christlich-soziale Partei Schweiz
7,https://politics.ld.admin.ch/party-version/65_...,https://politics.ld.admin.ch/party-version/66_...,NaN,Christlich-Soziale Partei der Schweiz
8,https://politics.ld.admin.ch/party-version/65_...,https://politics.ld.admin.ch/party-version/65_...,Christlichsoziale Partei der Schweiz,NaN
9,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...,Christlichdemokratische Volkspartei der Schweiz,Christlichdemokratische Volkspartei der Schweiz


In [10]:
from ipycytoscape import CytoscapeWidget
import networkx as nx

my_style = [
    {
        "selector": "node",
        "style": {
            "label": "data(name)",
            "background-color": "#EAF2F8",
            "color": "#1B2631",
            "text-valign": "center",
            "text-halign": "center",
            "font-size": 10,
            "shape": "rectangle",
            "width": "label",
            "height": 20,
            "padding": 10,
        },
    },
    {
        "selector": "edge.directed",
        "style": {
            "line-color": "#0074D9",
            "target-arrow-color": "#0074D9",
            "target-arrow-shape": "triangle",
            "curve-style": "bezier",
        },
    },
]

# replace empty names with ""
df['source_name'] = df['source_name'].fillna("no_name_available")
df['target_name'] = df['target_name'].fillna("no_name_available")

ids = pd.Series.tolist(df['source']) + pd.Series.tolist(df['target'])
names = pd.Series.tolist(df['source_name']) + pd.Series.tolist(df['target_name'])

node_df = pd.DataFrame({'id': ids, 'name': names})
node_df = node_df.drop_duplicates(subset=['id']).reset_index(drop=True)

G = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))

In [11]:
cyto = CytoscapeWidget()
cyto.graph.add_graph_from_networkx(G, directed=True)
cyto.set_style(my_style)
cyto.set_layout(name = "dagre", rankDir = "LR")

display(cyto)

CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'LR'}, cytoscape_style=[{'selector': 'node', 'st…

## Checker Queries

The following queries are used to check the data for inconsistencies. They can be used to find errors in the data.

### Identities without Versions

Expected result: No results

In [12]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?id WHERE {
    ?id a vl:Identity .
    FILTER NOT EXISTS {
        ?id vl:version ?version .
    }
}
    
"""

df = sparql_to_df(g, query)
display(df)

,id


### Versions with no Starting Class

Expected result: No results

In [13]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>
PREFIX pol: <https://politics.schema.ld.admin.ch/party-vocab/>

SELECT * WHERE {

    ?version a vl:Version ;

    FILTER NOT EXISTS {
        ?version pol:startedBy ?starting_class .
    }
} 
    
"""

df = sparql_to_df(g, query)
display(df)

,starting_class,version


### Versions with no Stopping Class

Expected result: No results

In [14]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>
PREFIX pol: <https://politics.schema.ld.admin.ch/party-vocab/>

SELECT * WHERE {

    ?version a vl:Version ;
        schema:validThrough ?validThrough .

    FILTER NOT EXISTS {
        ?version pol:stoppedBy ?stopping_class .
    }
} 
    
"""

df = sparql_to_df(g, query)
display(df)

,version,stopping_class,validThrough


### Versions with Stopping Class but no schema:validThrough

    Should not happen

In [15]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>
PREFIX pol: <https://politics.schema.ld.admin.ch/party-vocab/>

SELECT * WHERE {

    ?version a vl:Version ;
        pol:stoppedBy ?stopping_class .

    FILTER NOT EXISTS {
        ?version schema:validThrough ?validThrough .
    }
} 
    
"""

df = sparql_to_df(g, query)
display(df)

,version,stopping_class,validThrough


### Parties without Names

Expected result: No results

In [16]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?id WHERE {
    ?id a vl:Identity .
    FILTER NOT EXISTS {
        ?id schema:name ?name .
    }
}
    
"""

df = sparql_to_df(g, query)
display(df)

,id
0,https://politics.ld.admin.ch/party-id/353
1,https://politics.ld.admin.ch/party-id/135
2,https://politics.ld.admin.ch/party-id/158
3,https://politics.ld.admin.ch/party-id/194
4,https://politics.ld.admin.ch/party-id/226
5,https://politics.ld.admin.ch/party-id/250
6,https://politics.ld.admin.ch/party-id/258
7,https://politics.ld.admin.ch/party-id/260
8,https://politics.ld.admin.ch/party-id/262
9,https://politics.ld.admin.ch/party-id/304
